# 05-10 Fallbacks: 오류 발생 시 자동으로 대체 경로 사용하기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- `with_fallbacks()`로 기본 체인이 실패했을 때 자동으로 대체 체인으로 전환하는 폴백(Fallback) 패턴을 구현할 수 있어요
- 여러 폴백 체인을 순차적으로 지정하여 단계적 복구 전략을 설계할 수 있어요
- `exceptions_to_handle`로 특정 예외만 폴백으로 처리하고 나머지는 그대로 발생시키는 방법을 설명할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- 파이썬 예외 처리(`try/except`) 기초
- `ChatOpenAI` 모델 초기화와 `max_retries` 파라미터
- LCEL 파이프 연산자(`|`) 사용법

---

LLM 애플리케이션에는 다양한 오류가 발생할 수 있어요:

- LLM API 오류 (Rate Limit, 서버 장애 등)
- 모델 출력 품질 저하
- 통합 관련 이슈

`fallback` 기능을 사용하면 이러한 문제를 자동으로 감지하고 대체 경로로 우회할 수 있어요.

아래 다이어그램은 폴백 체인의 실행 흐름을 보여줘요:

```mermaid
flowchart TD
    IN["입력"]:::input
    P["Primary 모델<br/>기본 실행"]:::process
    F1["Fallback 1<br/>첫 번째 대체 모델"]:::error
    F2["Fallback 2<br/>두 번째 대체 모델"]:::error
    F3["Fallback 3<br/>세 번째 대체 모델"]:::error
    OK["성공<br/>응답 반환"]:::output
    ERR["최종 실패<br/>예외 발생"]:::error

    IN --> P
    P -- "성공" --> OK
    P -- "실패" --> F1
    F1 -- "성공" --> OK
    F1 -- "실패" --> F2
    F2 -- "성공" --> OK
    F2 -- "실패" --> F3
    F3 -- "성공" --> OK
    F3 -- "실패" --> ERR

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
```

**중요:** 폴백은 LLM 수준뿐만 아니라 전체 실행 가능한 체인 수준에 적용할 수 있어요.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 기본 모델 (재시도 비활성화)
primary_llm = ChatOpenAI(max_retries=0, model="gpt-4o-mini")

# Fallback 모델 (다른 모델 또는 서비스)
# 예제에서는 동일한 모델을 사용하지만, 실제로는 다른 모델을 사용
fallback_llm = ChatOpenAI(max_retries=0, model="gpt-3.5-turbo")

print("=" * 60)
print("✅ 모델 초기화 완료")
print("=" * 60)


✅ 모델 초기화 완료


## 1. 기본 Fallback 설정

`with_fallbacks()`를 사용하여 기본 모델이 실패할 때 대체 모델을 사용하도록 설정해요.

> **실무 팁:** 기본 모델에는 `max_retries=0`으로 설정하는 것을 권장해요. LangChain의 기본 재시도 로직과 폴백 로직이 중복되면 불필요한 대기 시간이 발생할 수 있어요. 폴백을 통해 재시도 전략을 명시적으로 제어하는 것이 더 효율적이에요.

In [4]:
# ============================================================
# TODO: primary_llm.with_fallbacks([fallback_llm])으로 폴백을 설정하세요
# 힌트: max_retries=0 권장 — 재시도와 폴백 로직이 중복되지 않도록
# 예상 결과: 기본 모델 실패 시 fallback_llm이 자동으로 실행
# ============================================================

# ---------------------------------------------------
# with_fallbacks([...]): 기본 체인 실패 시 대체 체인 순차 시도
# ---------------------------------------------------
llm_with_fallback = primary_llm.with_fallbacks([fallback_llm])

In [5]:
# Fallback이 설정된 모델 실행
# 
# 실제 오류가 발생하지 않으므로 정상 실행됨
res = llm_with_fallback.invoke("대한민국의 수도는 어디야?")
print(f' ==> [Line 4]: \033[38;2;50;90;143m[res]\033[0m({type(res).__name__}) = \033[38;2;146;250;90m{res}\033[0m')


 ==> [Line 4]: [res](AIMessage) = content='대한민국의 수도는 서울입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5714259854', 'id': 'chatcmpl-DPGwEoeZGWRevwekv8hNhzXhdTjVW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--019d4137-fcee-7d10-8ad3-b3e5583a1d1c-0' usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 2. 특정 예외만 처리하기

`exceptions_to_handle` 파라미터를 사용하여 특정 예외만 폴백으로 처리할 수 있어요.

나머지 예외는 폴백 없이 그대로 발생해요. 예를 들어 Rate Limit 오류만 폴백으로 처리하고 인증 오류는 그대로 발생시켜 명확하게 실패 원인을 파악할 수 있어요.

In [6]:
# 특정 예외만 처리하는 Fallback 설정
# 
# 시나리오: KeyboardInterrupt만 fallback으로 처리
#           다른 예외는 그대로 발생시킴

llm_selective_fallback = primary_llm.with_fallbacks(
    [fallback_llm],
    exceptions_to_handle=(KeyboardInterrupt,),  # 특정 예외만 처리
)

print("=" * 60)
print("✅ 선택적 Fallback 설정 완료")
print("=" * 60)
print("KeyboardInterrupt만 fallback으로 처리됩니다.")


✅ 선택적 Fallback 설정 완료
KeyboardInterrupt만 fallback으로 처리됩니다.


## 3. 여러 Fallback 모델 순차 지정

여러 개의 폴백 모델을 지정하면 순차적으로 시도해요.

첫 번째 모델 실패 → 두 번째 모델 시도 → 세 번째 모델 시도 → ...

> **주의:** 폴백 목록의 앞에는 비용이 낮은 모델을, 뒤에는 더 신뢰할 수 있는 모델을 배치하는 것이 일반적이에요. 비용과 신뢰성 사이의 균형을 고려해 폴백 순서를 설계하세요.

In [ ]:
# ============================================================
# TODO: 여러 fallback 모델을 순차적으로 지정하는 체인을 구성하세요
# 힌트: bad_model.with_fallbacks([fallback1, fallback2, fallback3])
# 예상 결과: bad_model 실패 → fallback_chain1 시도 → 성공 시 결과 반환
# ============================================================

# ---------------------------------------------------
# 여러 폴백 모델: 순차 시도로 단계적 복구
# ---------------------------------------------------


In [ ]:
# 프롬프트와 함께 사용하는 예제


## 마무리 요약

### with_fallbacks() 주요 파라미터

| 파라미터 | 역할 | 예시 |
|----------|------|------|
| 첫 번째 인자 (리스트) | 순서대로 시도할 폴백 체인 목록 | `[fallback1, fallback2]` |
| `exceptions_to_handle` | 폴백을 트리거할 예외 타입 | `(RateLimitError,)` |

### 핵심 요점

- `with_fallbacks([...])`로 기본 체인이 실패했을 때 자동으로 대체 체인을 순차적으로 시도해요
- 기본 모델에 `max_retries=0`을 설정하면 폴백 로직과 재시도 로직의 충돌을 방지해요
- `exceptions_to_handle`로 특정 예외만 폴백으로 처리하고 나머지는 그대로 전파할 수 있어요
- 폴백은 LLM 수준뿐 아니라 전체 체인(`prompt | llm | parser`) 수준에도 적용할 수 있어요

### 05_Advanced-LCEL 전체 요약

이 챕터에서 배운 LCEL 고급 기능들을 정리해요:

| 노트북 | 핵심 기능 | 주요 사용 사례 |
|--------|-----------|----------------|
| 01 RunnablePassthrough | 데이터 흐름 유지 + 키 추가 | RAG 질문 전달 |
| 02 Inspect Runnables | 체인 구조 시각화 | 디버깅, 구조 확인 |
| 03 RunnableLambda/@chain | 함수를 체인으로 변환 | 전처리/후처리 로직 |
| 04 Routing | 조건부 분기 | 다분야 챗봇 |
| 05 RunnableParallel | 병렬 실행 | 성능 최적화 |
| 06 Configure | 런타임 설정 변경 | A/B 테스트, 다중 모델 |
| 07 MessageHistory | 대화 기록 관리 | 멀티턴 챗봇 |
| 08 Custom Generator | 스트리밍 데이터 변환 | 실시간 출력 파서 |
| 09 Binding | 파라미터/도구 사전 연결 | Function Calling |
| 10 Fallbacks | 오류 복구 전략 | 안정적 서비스 |